# India Tech Job Market Analysis — Step 3: Data Cleaning & Preparation
This notebook prepares the dataset for SQL analysis and Power BI. The original CSV is not modified.

In [ ]:
import pandas as pd
import numpy as np

FILE_PATH = 'india_job_market_2024_2026.csv'
df = pd.read_csv(FILE_PATH)
print('Dataset loaded successfully')
print('Rows:', len(df))
print('Columns:', len(df.columns))

## 1. Clean text columns

In [ ]:
text_columns = df.select_dtypes(include='object').columns
for col in text_columns:
    df[col] = df[col].str.strip()
print('Text columns cleaned:', list(text_columns))

## 2. Convert dates and numeric columns

In [ ]:
df['Date_Posted'] = pd.to_datetime(df['Date_Posted'], errors='coerce')
numeric_columns = ['Salary_LPA', 'Openings', 'Applicants', 'Company_Rating']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print('Date range:', df['Date_Posted'].min().date(), 'to', df['Date_Posted'].max().date())
print('Missing values after conversion:')
print(df[['Date_Posted'] + numeric_columns].isna().sum())

## 3. Create analysis columns

In [ ]:
df['Applicants_per_Opening'] = df['Applicants'] / df['Openings']
df['Year'] = df['Date_Posted'].dt.year
df['Month'] = df['Date_Posted'].dt.month
df['Year_Month'] = df['Date_Posted'].dt.to_period('M').astype(str)
df['Skill_Count'] = df['Skills_Required'].fillna('').apply(lambda x: len([s.strip() for s in str(x).split(',') if s.strip()]))
display(df[['Applicants','Openings','Applicants_per_Opening','Skill_Count','Year','Month','Year_Month']].head())

## 4. Flag suspicious salary records
We do not automatically delete low-salary rows. We flag them so the records remain traceable.

In [ ]:
df['Salary_Anomaly_Flag'] = df['Salary_LPA'] <= 1
print('Potential salary anomalies:', int(df['Salary_Anomaly_Flag'].sum()))
display(df.loc[df['Salary_Anomaly_Flag'], ['Job_ID','Job_Title','Experience_Level','Salary_LPA','Company']].head(20))

## 5. Final quality checks

In [ ]:
print('Shape:', df.shape)
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate Job_IDs:', df['Job_ID'].duplicated().sum())
print('Missing values:', int(df.isna().sum().sum()))
print('Invalid openings:', int((df['Openings'] <= 0).sum()))
print('Invalid applicants:', int((df['Applicants'] < 0).sum()))
print('Applicants below openings:', int((df['Applicants'] < df['Openings']).sum()))

## 6. Save the cleaned dataset

In [ ]:
OUTPUT_FILE = 'india_job_market_2024_2026_cleaned.csv'
df.to_csv(OUTPUT_FILE, index=False)
print(f'Cleaned dataset saved as: {OUTPUT_FILE}')
print(f'Rows saved: {len(df)}')
print(f'Columns saved: {len(df.columns)}')

### Step 3 complete
The cleaned CSV is ready for the next stage: SQL analysis.